# 01: Exploratory Data Analysis

This notebook loads and inspects the BIS datasets for the FX prediction project.

We begin by:
1. Loading raw BIS files from `data/raw/`
2. Inspecting shapes, date ranges, and missing values
3. Identifying which currencies and economies are available
4. Checking data quality and alignment across series
5. Saving preliminary insights for downstream feature engineering


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Set up plotting defaults
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

# Define paths
ROOT = Path('.').resolve()
DATA_RAW = ROOT / 'data' / 'raw'
DATA_PROCESSED = ROOT / 'data' / 'processed'

print(f"Raw data directory: {DATA_RAW}")
print(f"Processed data directory: {DATA_PROCESSED}")
print(f"Raw data files: {list(DATA_RAW.glob('*'))}")

## 1. Load BIS Exchange Rate Data (WS_XRU)

Load the target variable: end-of-month USD exchange rates for 39 currencies.

In [ ]:
# Try to load WS_XRU with flexible extensions
xru_files = list(DATA_RAW.glob('WS_XRU*'))
print(f"Found WS_XRU files: {xru_files}")

if xru_files:
    xru_file = xru_files[0]
    print(f"Loading {xru_file.name}...")
    
    # Try CSV first, then Excel
    if xru_file.suffix.lower() in ['.csv', '.txt']:
        xru = pd.read_csv(xru_file, index_col=0, parse_dates=True)
    else:
        xru = pd.read_excel(xru_file, index_col=0, parse_dates=True)
    
    print(f"\nWS_XRU shape: {xru.shape}")
    print(f"Date range: {xru.index.min()} to {xru.index.max()}")
    print(f"\nFirst few rows:")
    print(xru.head())
    print(f"\nData types:")
    print(xru.dtypes)
else:
    print("WS_XRU file not found. Run 'python scripts/download_data.py' first.")

In [ ]:
# Inspect missing values
if 'xru' in locals():
    missing_counts = xru.isnull().sum()
    missing_pct = 100 * xru.isnull().sum() / len(xru)
    
    print("Missing values per currency:")
    missing_df = pd.DataFrame({
        'count': missing_counts,
        'percent': missing_pct
    }).sort_values('percent', ascending=False)
    print(missing_df)
    
    print(f"\nCurrencies with complete data: {(missing_df['count'] == 0).sum()}")

## 2. Load BIS Effective Exchange Rate Data (WS_EER)

Broad-based effective exchange rates for 63 economies, useful for context and robustness.

In [ ]:
eer_files = list(DATA_RAW.glob('WS_EER*'))
print(f"Found WS_EER files: {eer_files}")

if eer_files:
    eer_file = eer_files[0]
    print(f"Loading {eer_file.name}...")
    
    if eer_file.suffix.lower() in ['.csv', '.txt']:
        eer = pd.read_csv(eer_file, index_col=0, parse_dates=True)
    else:
        eer = pd.read_excel(eer_file, index_col=0, parse_dates=True)
    
    print(f"\nWS_EER shape: {eer.shape}")
    print(f"Date range: {eer.index.min()} to {eer.index.max()}")
    print(f"Number of economies: {eer.shape[1]}")
else:
    print("WS_EER file not found.")

## 3. Load BIS Central Bank Policy Rates (WS_CBPOL)

Policy rates for ~34 of the 39 target currencies.

In [ ]:
cbpol_files = list(DATA_RAW.glob('*CBPOL*'))
print(f"Found CBPOL files: {cbpol_files}")

if cbpol_files:
    cbpol_file = cbpol_files[0]
    print(f"Loading {cbpol_file.name}...")
    
    if cbpol_file.suffix.lower() in ['.csv', '.txt']:
        cbpol = pd.read_csv(cbpol_file, index_col=0, parse_dates=True)
    else:
        cbpol = pd.read_excel(cbpol_file, index_col=0, parse_dates=True)
    
    print(f"\nWS_CBPOL shape: {cbpol.shape}")
    print(f"Date range: {cbpol.index.min()} to {cbpol.index.max()}")
    print(f"Number of economies: {cbpol.shape[1]}")
else:
    print("CBPOL file not found.")

## 4. Load BIS Consumer Price Indices (WS_LONG_CPI)

Long-span consumer prices for ~62 economies.

In [ ]:
cpi_files = list(DATA_RAW.glob('*CPI*'))
print(f"Found CPI files: {cpi_files}")

if cpi_files:
    cpi_file = cpi_files[0]
    print(f"Loading {cpi_file.name}...")
    
    if cpi_file.suffix.lower() in ['.csv', '.txt']:
        cpi = pd.read_csv(cpi_file, index_col=0, parse_dates=True)
    else:
        cpi = pd.read_excel(cpi_file, index_col=0, parse_dates=True)
    
    print(f"\nWS_LONG_CPI shape: {cpi.shape}")
    print(f"Date range: {cpi.index.min()} to {cpi.index.max()}")
    print(f"Number of economies: {cpi.shape[1]}")
else:
    print("CPI file not found.")

## 5. Load BIS Global Liquidity Indicators (WS_GLI)

Dollar credit to non-banks outside the US, quarterly from 2000. This is a key global variable.

In [ ]:
gli_files = list(DATA_RAW.glob('*GLI*'))
print(f"Found GLI files: {gli_files}")

if gli_files:
    gli_file = gli_files[0]
    print(f"Loading {gli_file.name}...")
    
    if gli_file.suffix.lower() in ['.csv', '.txt']:
        gli = pd.read_csv(gli_file, index_col=0, parse_dates=True)
    else:
        gli = pd.read_excel(gli_file, index_col=0, parse_dates=True)
    
    print(f"\nWS_GLI shape: {gli.shape}")
    print(f"Date range: {gli.index.min()} to {gli.index.max()}")
    print(f"Variables available: {gli.shape[1]}")
    print(f"\nFirst few rows:")
    print(gli.head())
else:
    print("GLI file not found.")

## 6. Summary: Data Availability and Coverage

Now that we've inspected all datasets, let's summarize what we have for the panel construction.

In [ ]:
print("\n" + "="*70)
print("DATA AVAILABILITY SUMMARY")
print("="*70)

if 'xru' in locals():
    print(f"\nWS_XRU (Target: USD exchange rates)")
    print(f"  Shape: {xru.shape}")
    print(f"  Date range: {xru.index.min().date()} to {xru.index.max().date()}")
    print(f"  Frequency: Monthly")
    print(f"  Complete currencies: {(xru.isnull().sum() == 0).sum()} / {xru.shape[1]}")

if 'eer' in locals():
    print(f"\nWS_EER (Effective exchange rates)")
    print(f"  Shape: {eer.shape}")
    print(f"  Date range: {eer.index.min().date()} to {eer.index.max().date()}")
    print(f"  Frequency: Monthly")

if 'cbpol' in locals():
    print(f"\nWS_CBPOL (Central bank policy rates)")
    print(f"  Shape: {cbpol.shape}")
    print(f"  Date range: {cbpol.index.min().date()} to {cbpol.index.max().date()}")
    print(f"  Frequency: Monthly")

if 'cpi' in locals():
    print(f"\nWS_LONG_CPI (Consumer price indices)")
    print(f"  Shape: {cpi.shape}")
    print(f"  Date range: {cpi.index.min().date()} to {cpi.index.max().date()}")
    print(f"  Frequency: Monthly")

if 'gli' in locals():
    print(f"\nWS_GLI (Global liquidity indicators)")
    print(f"  Shape: {gli.shape}")
    print(f"  Date range: {gli.index.min().date()} to {gli.index.max().date()}")
    print(f"  Frequency: Quarterly")

print("\n" + "="*70)

## 7. Identify Target Currencies and Data Quality Issues

Explore which currencies are available in WS_XRU and understand common data quality patterns.

In [ ]:
if 'xru' in locals():
    print(f"Available currencies in WS_XRU: ({xru.shape[1]})")
    print("\nCurrency codes:")
    for i, col in enumerate(xru.columns, 1):
        print(f"  {i:2d}. {col}")
    
    # Show summary statistics
    print("\nSummary statistics (sample of currencies):")
    print(xru.iloc[:, :5].describe())

## 8. Visualize Data Coverage

Create a heatmap of non-missing data across time and currencies.

In [ ]:
if 'xru' in locals():
    # Create a heatmap of data availability
    fig, ax = plt.subplots(figsize=(14, 8))
    
    # Sample every Nth year for readability
    sample_freq = 12  # every 12 months (1 year)
    data_availability = (~xru.isnull()).astype(int).iloc[::sample_freq, :]
    
    sns.heatmap(data_availability.T, cbar_kws={'label': 'Data available'}, 
                cmap='RdYlGn', ax=ax, xticklabels=True)
    ax.set_xlabel('Time (yearly ticks)\nJanuary 1994 – August 2026')
    ax.set_ylabel('Currency')
    ax.set_title('WS_XRU Data Availability by Currency and Time')
    plt.tight_layout()
    plt.savefig(DATA_PROCESSED / 'xru_data_availability.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print(f"Plot saved to {DATA_PROCESSED / 'xru_data_availability.png'}")

## 9. Check Currency Overlap Across Datasets

Which currencies appear in both WS_XRU (target) and WS_CBPOL (interest rates)?

In [ ]:
if 'xru' in locals() and 'cbpol' in locals():
    xru_curs = set(xru.columns)
    cbpol_curs = set(cbpol.columns)
    
    overlap = xru_curs & cbpol_curs
    xru_only = xru_curs - cbpol_curs
    cbpol_only = cbpol_curs - xru_curs
    
    print(f"\nCurrency overlap between WS_XRU and WS_CBPOL:")
    print(f"  In both: {len(overlap)}")
    print(f"  In WS_XRU only: {len(xru_only)}")
    print(f"  In WS_CBPOL only: {len(cbpol_only)}")
    print(f"\nCurrencies in both datasets: {sorted(overlap)}")

## 10. Notes for Next Steps

Based on this exploratory analysis, note down:
- Which currencies you will exclude (dollar pegs, renminbi/ringgit peg periods)
- Date range for the modeling sample (e.g., Jan 2010 onward for expanding window)
- Which interest-rate and CPI proxies to use for currencies with missing CBPOL
- How to align quarterly GLI data with monthly observations

This notebook is the foundation for the feature-engineering step in **02_feature_engineering.ipynb**.

In [ ]:
print("\nExploration complete. Move on to 02_feature_engineering.ipynb to build the panel.")